# 그래디언트 부스팅 실습

**Gradient Boosting · XGBoost · LightGBM**

앞선 모델의 잔차를 순차적으로 보완하는 트리를 더해 성능을 높이는 앙상블 방법.

소재 분야에서 이해하기: 표 형태의 공정 데이터에서 수율을 예측한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 앙상블 문서](https://scikit-learn.org/stable/modules/ensemble.html)

## 1. 잔차를 순차적으로 보완

부스팅이 단계마다 무엇을 고치는지 눈으로 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

x = np.sort(rng.uniform(0, 1, 160))[:, None]
y_true = np.sin(3 * x[:, 0]) + 0.4 * x[:, 0] ** 2
y_obs = y_true + rng.normal(0, 0.08, x.shape[0])

In [ ]:
from sklearn.tree import DecisionTreeRegressor

prediction = np.full(x.shape[0], y_obs.mean())
learning_rate = 0.4
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for step, axis in enumerate(axes, 1):
    for _ in range(step * 3):
        residual = y_obs - prediction
        stump = DecisionTreeRegressor(max_depth=2, random_state=0).fit(x, residual)
        prediction = prediction + learning_rate * stump.predict(x)
    axis.scatter(x[:, 0], y_obs, s=8, alpha=0.5)
    axis.plot(x[:, 0], prediction, 'r-')
    axis.set_title('%d trees' % (step * 3))
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import cross_val_score

X_full, y_full = X, y
for name, model in [('랜덤 포레스트', RandomForestRegressor(n_estimators=300, random_state=0)),
                    ('부스팅(lr=0.1)', GradientBoostingRegressor(random_state=0)),
                    ('부스팅(lr=1.0)', GradientBoostingRegressor(learning_rate=1.0, random_state=0))]:
    scores = cross_val_score(model, X_full, y_full, cv=5, scoring='neg_mean_absolute_error')
    print('%-16s MAE %.2f ± %.2f' % (name, -scores.mean(), scores.std()))

## 2. 해석

부스팅은 약한 모델을 순차적으로 더해 잔차를 줄입니다. 학습률이 크면 빨리 맞추지만 과적합하기 쉽고,
작으면 트리를 많이 필요로 합니다. 표 형태의 소재 데이터에서 자주 가장 좋은 성능을 냅니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#gradient-boosting)을 여세요.